# TreeNAM: Additive differentiable trees

TreeNAM assigns one soft neural decision tree to each feature and optional interaction, retaining a term-wise additive output.


## Model


For a depth-$D$ soft tree,

$$
f_j(x_j)=\sum_{\ell=1}^{2^D}p_{j\ell}(x_j)v_{j\ell},
\qquad
\eta(x)=\beta_0+\sum_jf_j(x_j)+\sum_{S\in\mathcal I}f_S(x_S).
$$

$p_{j\ell}$ is the product of differentiable routing probabilities along a path.


## Shared estimator API

All neural estimators use `fit`, `predict`, `score`, `evaluate`, and
`predict_components`. The component result reconstructs predictions on the link
scale and supports shared term-importance and plotting utilities. Constructor
options such as `numerical_preprocessing` and `categorical_preprocessing` are forwarded to
PreTab and are fitted on training rows only.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(7)
n = 180
X = pd.DataFrame({
    "x1": rng.uniform(-1.0, 1.0, n),
    "x2": rng.normal(size=n),
    "group": rng.choice(["a", "b", "c"], size=n),
})
y = (
    np.sin(np.pi * X["x1"])
    + 0.35 * X["x2"] ** 2
    + 0.30 * (X["group"] == "b")
    + rng.normal(0.0, 0.12, n)
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=7
)

# Set True to run the small fit and all fitted-model demonstrations.
RUN_TRAINING = False


## Construct the estimator


In [ ]:
from nampy.models import TreeNAMClassifier, TreeNAMLSS, TreeNAMRegressor


model = TreeNAMRegressor(
    tree_depth=3,
    tree_lamda=1e-3,
    tree_temperature=1.0,
    use_hard_routing_in_eval=False,
    interactions=(("x1", "x2"),),
)
model.get_params(deep=False)


## Fit and inspect

Enable `RUN_TRAINING` above for a short demonstration. Real work should use a
larger validation set, enough epochs, and early stopping.


In [ ]:
if RUN_TRAINING:
    model.fit(
        X_train,
        y_train,
        max_epochs=3,
        batch_size=64,
        random_state=7,
        logger=False,
        enable_progress_bar=False,
        enable_model_summary=False,
    )
    predictions = model.predict(X_test)
    r2 = model.score(X_test, y_test)
    metrics = model.evaluate(X_test, y_test)
    components = model.predict_components(X_test, center=True)
    components.validate_additive_reconstruction()
    display({"R2": r2, **metrics})
    display(model.term_importance(X_test).head())


## Model-specific controls

`tree_temperature` controls soft routing; `use_hard_routing_in_eval=True` selects a single leaf path at inference. The tree penalty is included during training.


In [ ]:
hard_routing_model = TreeNAMRegressor(
    tree_depth=3,
    tree_temperature=1.0,
    use_hard_routing_in_eval=True,
)
if RUN_TRAINING:
    display(model.interaction_importance(X_test))
    display(model.model_complexity())


## Task variants and limits

TreeNAM supports regression, classification, and LSS objectives.
